# Import all python files and packages

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from clean_emi_data import *
from preprocess_emi_data import *
from clean_generation_output_data import *
from preprocessing_Anu import *

# define globale preprocessing variables

In [2]:
start_date = "2019-01-01" #used for cuttig the exact timespan
end_date   = "2024-12-31"

save_path = 'preprocessed_data.csv'

file_path_wind        = 'data_input/Wind_data_100m.csv'
file_path_demand      = 'data_input/demand_by_zone.csv'
file_path_lake        = 'data_input/Lakes storage levels'
file_path_solar       = 'data_input/Solar_data.csv'
file_path_temperature = 'data_input/Temperature_data.csv'
file_path_hvdc        = 'data_input/hvdc_transfer.csv'
file_path_outages     = 'data_input/scheduled_outages.csv'
file_path_holidays    = 'data_input/nz_holidays_onehot.csv'
file_path_generation  = 'data_input/generation_output_merged.csv'

# First df cut to length for left join later

# Wholesale Price

In [3]:
df = preprocess_wholesale_price(clean_wholesale_price())

Empty DataFrame
Columns: [el_price_dol/MWh_BEN2201, el_price_dol/MWh_HAY2201, el_price_dol/MWh_INV2201, el_price_dol/MWh_ISL2201, el_price_dol/MWh_KIK2201, el_price_dol/MWh_OTA2201, el_price_dol/MWh_RDF2201, el_price_dol/MWh_SFD2201, el_price_dol/MWh_WKM2201]
Index: []
✅ wholesale_price has been cleaned and saved under data_output/wholesale_price_utc12.csv ✅
✅ wholesale price has been preprocessed and saved under data_output/wholesale_price_preprocessed.csv ✅


In [4]:
df.head()

,datetime_utc12,el_price_dol_MWh_BEN2201,el_price_dol_MWh_HAY2201,el_price_dol_MWh_INV2201,el_price_dol_MWh_ISL2201,el_price_dol_MWh_KIK2201,el_price_dol_MWh_OTA2201,el_price_dol_MWh_RDF2201,el_price_dol_MWh_SFD2201,el_price_dol_MWh_WKM2201
0,2013-12-31 23:00:00,35.130,46.670,35.650,36.070,NaN,50.690,48.710,48.450,49.500
1,2014-01-01 00:00:00,33.755,34.900,34.265,34.720,NaN,37.845,36.420,36.195,36.865
2,2014-01-01 01:00:00,32.455,33.555,32.945,33.385,NaN,36.350,35.000,34.785,35.425
3,2014-01-01 02:00:00,5.555,5.745,5.635,5.710,NaN,6.215,5.975,5.950,6.050
4,2014-01-01 03:00:00,3.865,3.990,3.920,3.970,NaN,4.315,4.150,4.135,4.210


In [5]:
df = df[(df['datetime_utc12'] >= start_date) & (df['datetime_utc12'] <= end_date)].reset_index(drop=True)

In [6]:
df.head(-1)

,datetime_utc12,el_price_dol_MWh_BEN2201,el_price_dol_MWh_HAY2201,el_price_dol_MWh_INV2201,el_price_dol_MWh_ISL2201,el_price_dol_MWh_KIK2201,el_price_dol_MWh_OTA2201,el_price_dol_MWh_RDF2201,el_price_dol_MWh_SFD2201,el_price_dol_MWh_WKM2201
0,2019-01-01 00:00:00,169.985,171.760,180.715,181.835,184.255,176.515,166.425,169.515,167.185
1,2019-01-01 01:00:00,156.335,157.870,166.360,167.215,169.400,162.070,152.940,155.645,153.635
2,2019-01-01 02:00:00,139.650,141.000,148.550,149.095,151.050,144.645,136.500,138.990,137.120
3,2019-01-01 03:00:00,128.845,129.980,137.375,137.215,139.015,133.355,125.830,128.130,126.400
4,2019-01-01 04:00:00,134.865,136.140,144.585,143.745,145.670,139.955,132.130,134.465,132.725
...,...,...,...,...,...,...,...,...,...,...
52579,2024-12-30 19:00:00,9.000,9.530,9.010,9.415,9.665,10.765,9.950,10.060,10.060
52580,2024-12-30 20:00:00,9.000,9.530,9.010,9.420,9.605,10.795,9.965,10.000,10.075
52581,2024-12-30 21:00:00,4.635,4.885,4.640,4.855,4.915,5.620,5.190,5.125,5.250
52582,2024-12-30 22:00:00,4.375,4.365,4.565,4.610,4.665,4.710,4.425,4.445,4.480


# importing and merging the rest

# generation output

In [7]:
df_gen = preprocess_generation_data(file_path_generation, start_date, end_date)

In [8]:
df_gen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52929 entries, 0 to 52928
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   datetime_utc12  52929 non-null  datetime64[ns]
 1   Coal            52929 non-null  float64       
 2   Diesel          52929 non-null  float64       
 3   Ele             52929 non-null  float64       
 4   Gas             52929 non-null  float64       
 5   Geo             52929 non-null  float64       
 6   Hydro           52929 non-null  float64       
 7   Solar           52929 non-null  float64       
 8   Wind            52929 non-null  float64       
 9   Wood            52929 non-null  float64       
dtypes: datetime64[ns](1), float64(9)
memory usage: 4.0 MB


In [9]:
df = df.merge(df_gen, how='left', on='datetime_utc12')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52585 entries, 0 to 52584
Data columns (total 19 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   datetime_utc12            52585 non-null  datetime64[ns]
 1   el_price_dol_MWh_BEN2201  52585 non-null  float64       
 2   el_price_dol_MWh_HAY2201  52585 non-null  float64       
 3   el_price_dol_MWh_INV2201  52585 non-null  float64       
 4   el_price_dol_MWh_ISL2201  52585 non-null  float64       
 5   el_price_dol_MWh_KIK2201  52585 non-null  float64       
 6   el_price_dol_MWh_OTA2201  52585 non-null  float64       
 7   el_price_dol_MWh_RDF2201  52585 non-null  float64       
 8   el_price_dol_MWh_SFD2201  52585 non-null  float64       
 9   el_price_dol_MWh_WKM2201  52585 non-null  float64       
 10  Coal                      52567 non-null  float64       
 11  Diesel                    52567 non-null  float64       
 12  Ele               

# wind

In [11]:
df_wind = preprocess_wind_data(file_path_wind)

In [12]:
df_wind.head()

,datetime,palmerston_north_wind_kmh,palmerston_north_wind_dir_deg,wellington_wind_kmh,wellington_wind_dir_deg,harapaki_hawkesbay_wind_kmh,harapaki_hawkesbay_wind_dir_deg,te_uku_waikato_wind_kmh,te_uku_waikato_wind_dir_deg,kaiwera_downs_southland_wind_kmh,kaiwera_downs_southland_wind_dir_deg
0,2014-01-01 00:00:00,20.2,325,36.9,324,4.5,346,12.8,286,4.4,279
1,2014-01-01 01:00:00,20.9,336,36.3,323,5.9,317,8.6,285,1.5,284
2,2014-01-01 02:00:00,22.5,347,35.7,326,7.6,301,4.3,312,1.5,284
3,2014-01-01 03:00:00,23.1,355,36.3,328,8.5,298,5.6,345,1.1,72
4,2014-01-01 04:00:00,23.0,360,35.9,329,8.9,291,6.5,360,4.3,85


In [13]:
df =df.merge(df_wind, how='left', left_on='datetime_utc12', right_on='datetime')
df = df.drop(columns=['datetime'])

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52585 entries, 0 to 52584
Data columns (total 29 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   datetime_utc12                        52585 non-null  datetime64[ns]
 1   el_price_dol_MWh_BEN2201              52585 non-null  float64       
 2   el_price_dol_MWh_HAY2201              52585 non-null  float64       
 3   el_price_dol_MWh_INV2201              52585 non-null  float64       
 4   el_price_dol_MWh_ISL2201              52585 non-null  float64       
 5   el_price_dol_MWh_KIK2201              52585 non-null  float64       
 6   el_price_dol_MWh_OTA2201              52585 non-null  float64       
 7   el_price_dol_MWh_RDF2201              52585 non-null  float64       
 8   el_price_dol_MWh_SFD2201              52585 non-null  float64       
 9   el_price_dol_MWh_WKM2201              52585 non-null  float64       
 10

# Solar

In [15]:
df_solar = preprocess_solar_data(file_path_solar)

In [16]:
df_solar.head()

,datetime,auckland_shortwave_wm2,auckland_sunshine_s,christchurch_shortwave_wm2,christchurch_sunshine_s,wellington_shortwave_wm2,wellington_sunshine_s,hamilton_shortwave_wm2,hamilton_sunshine_s,tauranga_shortwave_wm2,tauranga_sunshine_s,dunedin_shortwave_wm2,dunedin_sunshine_s
0,2014-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2014-01-01 01:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2014-01-01 02:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2014-01-01 03:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2014-01-01 04:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
df =df.merge(df_solar, how='left', left_on='datetime_utc12', right_on='datetime')
df = df.drop(columns=['datetime'])

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52585 entries, 0 to 52584
Data columns (total 41 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   datetime_utc12                        52585 non-null  datetime64[ns]
 1   el_price_dol_MWh_BEN2201              52585 non-null  float64       
 2   el_price_dol_MWh_HAY2201              52585 non-null  float64       
 3   el_price_dol_MWh_INV2201              52585 non-null  float64       
 4   el_price_dol_MWh_ISL2201              52585 non-null  float64       
 5   el_price_dol_MWh_KIK2201              52585 non-null  float64       
 6   el_price_dol_MWh_OTA2201              52585 non-null  float64       
 7   el_price_dol_MWh_RDF2201              52585 non-null  float64       
 8   el_price_dol_MWh_SFD2201              52585 non-null  float64       
 9   el_price_dol_MWh_WKM2201              52585 non-null  float64       
 10

# Temperature

In [19]:
df_temp = preprocess_temperature_data(file_path_temperature)

In [20]:
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108672 entries, 0 to 108671
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   datetime             108672 non-null  datetime64[ns]
 1   auckland_temp_c      108672 non-null  float64       
 2   christchurch_temp_c  108672 non-null  float64       
 3   wellington_temp_c    108672 non-null  float64       
 4   hamilton_temp_c      108672 non-null  float64       
 5   tauranga_temp_c      108672 non-null  float64       
 6   dunedin_temp_c       108672 non-null  float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 5.8 MB


In [21]:
df =df.merge(df_temp, how='left', left_on='datetime_utc12', right_on='datetime')
df = df.drop(columns=['datetime'])

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52585 entries, 0 to 52584
Data columns (total 47 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   datetime_utc12                        52585 non-null  datetime64[ns]
 1   el_price_dol_MWh_BEN2201              52585 non-null  float64       
 2   el_price_dol_MWh_HAY2201              52585 non-null  float64       
 3   el_price_dol_MWh_INV2201              52585 non-null  float64       
 4   el_price_dol_MWh_ISL2201              52585 non-null  float64       
 5   el_price_dol_MWh_KIK2201              52585 non-null  float64       
 6   el_price_dol_MWh_OTA2201              52585 non-null  float64       
 7   el_price_dol_MWh_RDF2201              52585 non-null  float64       
 8   el_price_dol_MWh_SFD2201              52585 non-null  float64       
 9   el_price_dol_MWh_WKM2201              52585 non-null  float64       
 10

# Lake storage

In [23]:
df_test=make_lake_hourly(preprocess_lake_storage('data_input/Lakes storage levels/SI_WPU_Storage_LakeWakatipu.csv'))

In [24]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 394465 entries, 0 to 394464
Data columns (total 3 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   datetime            394465 non-null  datetime64[ns]
 1   lake_level_m        394465 non-null  float64       
 2   active_storage_mm³  394465 non-null  float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 9.0 MB


In [25]:

for filename in os.listdir(file_path_lake):
    filepath = os.path.join(file_path_lake, filename)
    prefix = filename[:7]

    df_lake = make_lake_hourly(preprocess_lake_storage(filepath), start_date, end_date)

    df_lake = df_lake.rename(columns={
        col: f'{prefix}_{col}'
        for col in df_lake.columns
        if col != 'datetime'
    })
    df = df.merge(df_lake, how='left', left_on='datetime_utc12', right_on='datetime')
    df = df.drop(columns=['datetime'])

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52585 entries, 0 to 52584
Data columns (total 67 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   datetime_utc12                        52585 non-null  datetime64[ns]
 1   el_price_dol_MWh_BEN2201              52585 non-null  float64       
 2   el_price_dol_MWh_HAY2201              52585 non-null  float64       
 3   el_price_dol_MWh_INV2201              52585 non-null  float64       
 4   el_price_dol_MWh_ISL2201              52585 non-null  float64       
 5   el_price_dol_MWh_KIK2201              52585 non-null  float64       
 6   el_price_dol_MWh_OTA2201              52585 non-null  float64       
 7   el_price_dol_MWh_RDF2201              52585 non-null  float64       
 8   el_price_dol_MWh_SFD2201              52585 non-null  float64       
 9   el_price_dol_MWh_WKM2201              52585 non-null  float64       
 10

# Holidays

In [27]:
df_holiday = create_nz_holiday_data(start_date, end_date)

In [28]:
df = df.merge(df_holiday, how='left', left_on='datetime_utc12', right_on='datetime')
df = df.drop(columns=['datetime'])

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52585 entries, 0 to 52584
Data columns (total 68 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   datetime_utc12                        52585 non-null  datetime64[ns]
 1   el_price_dol_MWh_BEN2201              52585 non-null  float64       
 2   el_price_dol_MWh_HAY2201              52585 non-null  float64       
 3   el_price_dol_MWh_INV2201              52585 non-null  float64       
 4   el_price_dol_MWh_ISL2201              52585 non-null  float64       
 5   el_price_dol_MWh_KIK2201              52585 non-null  float64       
 6   el_price_dol_MWh_OTA2201              52585 non-null  float64       
 7   el_price_dol_MWh_RDF2201              52585 non-null  float64       
 8   el_price_dol_MWh_SFD2201              52585 non-null  float64       
 9   el_price_dol_MWh_WKM2201              52585 non-null  float64       
 10

# Demand per zone

In [30]:
df_demand = preprocess_demand_per_zone(clean_demand_per_zone())

Empty DataFrame
Columns: [demand_GWh_CNI, demand_GWh_LNI, demand_GWh_LSI, demand_GWh_UNI, demand_GWh_USI]
Index: []
✅ demand has been cleaned and saved under data_output/demand_utc12.csv ✅
✅ demand has been preprocessed and saved under data_output/demand_preprocessed.csv ✅


In [31]:
df_demand.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108709 entries, 0 to 108708
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   datetime_utc12  108709 non-null  datetime64[ns]
 1   demand_GWh_CNI  108709 non-null  float64       
 2   demand_GWh_LNI  108709 non-null  float64       
 3   demand_GWh_LSI  108709 non-null  float64       
 4   demand_GWh_UNI  108709 non-null  float64       
 5   demand_GWh_USI  108709 non-null  float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 5.0 MB


In [32]:
df = df.merge(df_demand, how='left')

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52585 entries, 0 to 52584
Data columns (total 73 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   datetime_utc12                        52585 non-null  datetime64[ns]
 1   el_price_dol_MWh_BEN2201              52585 non-null  float64       
 2   el_price_dol_MWh_HAY2201              52585 non-null  float64       
 3   el_price_dol_MWh_INV2201              52585 non-null  float64       
 4   el_price_dol_MWh_ISL2201              52585 non-null  float64       
 5   el_price_dol_MWh_KIK2201              52585 non-null  float64       
 6   el_price_dol_MWh_OTA2201              52585 non-null  float64       
 7   el_price_dol_MWh_RDF2201              52585 non-null  float64       
 8   el_price_dol_MWh_SFD2201              52585 non-null  float64       
 9   el_price_dol_MWh_WKM2201              52585 non-null  float64       
 10

# HVDC

In [34]:
df_hvdc = preprocess_hvdc(clean_hvdc())

✅ hvdc has been cleaned and saved under data_output/hvdc_utc12.csv ✅
✅ hvdc has been preprocessed and saved under data_output/hvdc_preprocessed.csv ✅


In [35]:
df_hvdc.head()

,datetime_utc12,avg_flow_MW,peak_flow_MW,Direction
0,2013-12-31 23:00:00,445.0,445.0,1
1,2014-01-01 00:00:00,373.0,386.0,1
2,2014-01-01 01:00:00,366.0,367.0,1
3,2014-01-01 02:00:00,326.0,328.0,1
4,2014-01-01 03:00:00,293.5,319.0,1


In [36]:
df = df.merge(df_hvdc, how='left')

# Scheduled Outages

In [37]:
df_outages = preprocess_outages(clean_outages())

✅ production outages has been cleaned and saved under data_output/scheduled_outages_utc12.csv ✅
✅ outages has been preprocessed and saved under data_output/outages_preprocessed.csv ✅


In [38]:
df_outages.head()

,datetime_utc12,outage_Gas_MW,outage_Hyd_MW,outage_Ter_MW,outage_Win_MW,outage_UNKN_MW
0,2013-12-31 23:00:00,0.0,50.2,250.0,0.0,0.0
1,2014-01-01 00:00:00,0.0,50.2,250.0,0.0,0.0
2,2014-01-01 01:00:00,0.0,50.2,250.0,0.0,0.0
3,2014-01-01 02:00:00,0.0,50.2,250.0,0.0,0.0
4,2014-01-01 03:00:00,0.0,50.2,250.0,0.0,0.0


In [39]:
df=df.merge(df_outages, how='left')

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52585 entries, 0 to 52584
Data columns (total 81 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   datetime_utc12                        52585 non-null  datetime64[ns]
 1   el_price_dol_MWh_BEN2201              52585 non-null  float64       
 2   el_price_dol_MWh_HAY2201              52585 non-null  float64       
 3   el_price_dol_MWh_INV2201              52585 non-null  float64       
 4   el_price_dol_MWh_ISL2201              52585 non-null  float64       
 5   el_price_dol_MWh_KIK2201              52585 non-null  float64       
 6   el_price_dol_MWh_OTA2201              52585 non-null  float64       
 7   el_price_dol_MWh_RDF2201              52585 non-null  float64       
 8   el_price_dol_MWh_SFD2201              52585 non-null  float64       
 9   el_price_dol_MWh_WKM2201              52585 non-null  float64       
 10

In [41]:
df.to_csv(save_path, index=False)